# Decision Tree

**INDE 577 / CMOR 438 — Rice University**  
**Instructor:** Randy R. Davila, PhD

---

## Overview

Decision Trees are **non-parametric, interpretable** models that partition the feature space into axis-aligned rectangular regions using a series of binary splits. They can be used for both **classification** and **regression**.

## Mathematical Background

### Splitting Criteria

**Gini Impurity** (for classification):
$$G(S) = 1 - \sum_{k=1}^{K} p_k^2$$

**Entropy / Information Gain** (for classification):
$$H(S) = -\sum_{k=1}^{K} p_k \log_2 p_k$$

$$\text{Information Gain} = H(S) - \frac{|S_L|}{|S|} H(S_L) - \frac{|S_R|}{|S|} H(S_R)$$

**Variance Reduction** (for regression):
$$\text{Var. Reduction} = \text{Var}(S) - \frac{|S_L|}{|S|} \text{Var}(S_L) - \frac{|S_R|}{|S|} \text{Var}(S_R)$$

### CART Algorithm

At each node, find the best feature $j$ and threshold $t$:

$$\arg\min_{j, t} \left[ \frac{|S_L|}{|S|} \cdot \text{impurity}(S_L) + \frac{|S_R|}{|S|} \cdot \text{impurity}(S_R) \right]$$

### Stopping Criteria
- Maximum depth
- Minimum samples per split
- Minimum samples per leaf
- No impurity improvement

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris, load_breast_cancer, make_regression
from sklearn.tree import DecisionTreeClassifier as SklearnDT
import warnings
warnings.filterwarnings('ignore')

from rice_ml import DecisionTreeClassifier, DecisionTreeRegressor
from rice_ml.processing.preprocessing import StandardScaler, train_test_split
from rice_ml.processing.metrics import accuracy_score, classification_report

print("Libraries loaded!")
np.random.seed(42)

## 1. Iris Classification — Visualizing Decision Boundaries

In [ ]:
iris = load_iris()
# Use 2 features for visualization
X_2, y = iris.data[:, 2:4], iris.target  # petal length, petal width
feature_names = ['Petal Length (cm)', 'Petal Width (cm)']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
depths = [2, 4, None]  # None = unlimited
colors = ['#e74c3c', '#3498db', '#2ecc71']

x_min, x_max = X_2[:, 0].min() - 0.3, X_2[:, 0].max() + 0.3
y_min, y_max = X_2[:, 1].min() - 0.3, X_2[:, 1].max() + 0.3
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300))

for ax, depth in zip(axes, depths):
    dt = DecisionTreeClassifier(max_depth=depth, criterion='gini')
    dt.fit(X_2, y)
    Z = dt.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdYlGn', levels=[-0.5, 0.5, 1.5, 2.5])
    for i, color in enumerate(colors):
        mask = y == i
        ax.scatter(X_2[mask, 0], X_2[mask, 1], c=color, s=40, alpha=0.8,
                   edgecolors='k', lw=0.4, label=iris.target_names[i])
    
    acc = accuracy_score(y, dt.predict(X_2))
    depth_str = str(depth) if depth else 'unlimited'
    ax.set_title(f'Depth={depth_str}\n(acc={acc:.2f})', fontsize=13, fontweight='bold')
    ax.set_xlabel(feature_names[0], fontsize=10)
    ax.set_ylabel(feature_names[1], fontsize=10)
    if depth == depths[0]:
        ax.legend(fontsize=9)

plt.suptitle('Decision Tree Boundaries: Iris (Petal Features)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('decision_tree_iris.png', dpi=120, bbox_inches='tight')
plt.show()

## 2. Breast Cancer Classification

In [ ]:
data = load_breast_cancer()
X, y = data.data, data.target
feature_names = data.feature_names
target_names = data.target_names

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

dt = DecisionTreeClassifier(max_depth=5, criterion='gini', min_samples_split=2)
dt.fit(X_train, y_train)
y_pred = dt.predict(X_test)

sk_dt = SklearnDT(max_depth=5, criterion='gini', random_state=42)
sk_dt.fit(X_train, y_train)

print("=== Decision Tree (depth=5) on Breast Cancer ===")
print(classification_report(y_test, y_pred, target_names=list(target_names)))
print(f"rice_ml accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"sklearn accuracy:  {accuracy_score(y_test, sk_dt.predict(X_test)):.4f}")

In [ ]:
# Depth vs accuracy
depths_range = range(1, 16)
train_accs, test_accs = [], []

for d in depths_range:
    m = DecisionTreeClassifier(max_depth=d, criterion='gini')
    m.fit(X_train, y_train)
    train_accs.append(accuracy_score(y_train, m.predict(X_train)))
    test_accs.append(accuracy_score(y_test, m.predict(X_test)))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(list(depths_range), train_accs, 'bo-', linewidth=2, markersize=5, label='Train')
ax.plot(list(depths_range), test_accs, 'rs--', linewidth=2, markersize=5, label='Test')
ax.set_xlabel('Max Tree Depth', fontsize=11)
ax.set_ylabel('Accuracy', fontsize=11)
ax.set_title('Bias-Variance Tradeoff vs Depth', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Feature importance
ax2 = axes[1]
if hasattr(dt, 'feature_importances_'):
    importances = dt.feature_importances_
    top_idx = np.argsort(importances)[-10:][::-1]
    ax2.barh(range(10), importances[top_idx], color='steelblue', edgecolor='k', lw=0.5)
    ax2.set_yticks(range(10))
    ax2.set_yticklabels([feature_names[i][:22] for i in top_idx], fontsize=8)
    ax2.set_xlabel('Gini Importance', fontsize=11)
    ax2.set_title('Top 10 Feature Importances', fontsize=13, fontweight='bold')
    ax2.grid(True, alpha=0.3, axis='x')
else:
    # Alternative: show criterion comparison
    for crit, color in [('gini', 'steelblue'), ('entropy', 'crimson')]:
        accs = []
        for d in depths_range:
            m = DecisionTreeClassifier(max_depth=d, criterion=crit)
            m.fit(X_train, y_train)
            accs.append(accuracy_score(y_test, m.predict(X_test)))
        ax2.plot(list(depths_range), accs, 'o-', color=color, linewidth=2, markersize=5, label=crit)
    ax2.set_xlabel('Depth', fontsize=11)
    ax2.set_ylabel('Test Accuracy', fontsize=11)
    ax2.set_title('Gini vs Entropy Criterion', fontsize=13, fontweight='bold')
    ax2.legend(fontsize=10)
    ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('decision_tree_cancer.png', dpi=120, bbox_inches='tight')
plt.show()

## 3. Decision Tree Regression

In [ ]:
np.random.seed(42)
X_r = np.sort(np.random.uniform(0, 2*np.pi, 150)).reshape(-1, 1)
y_r = np.sin(X_r.ravel()) + np.random.normal(0, 0.2, 150)

X_line = np.linspace(0, 2*np.pi, 500).reshape(-1, 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, depth in zip(axes, [2, 5, 10]):
    dtr = DecisionTreeRegressor(max_depth=depth)
    dtr.fit(X_r, y_r)
    y_line = dtr.predict(X_line)
    
    ax.scatter(X_r, y_r, c='steelblue', s=25, alpha=0.6, edgecolors='k', lw=0.3, label='Data')
    ax.plot(X_line, np.sin(X_line), 'k--', linewidth=2, alpha=0.7, label='True: sin(x)')
    ax.plot(X_line, y_line, 'r-', linewidth=2.5, label=f'Tree (depth={depth})')
    ax.set_xlabel('x', fontsize=11)
    ax.set_ylabel('y', fontsize=11)
    ax.set_title(f'Decision Tree Regressor\n(depth={depth})', fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('decision_tree_regression.png', dpi=120, bbox_inches='tight')
plt.show()

## Summary

| Property | Value |
|---|---|
| Type | Non-parametric, rule-based |
| Training | Greedy recursive splitting (CART) |
| Splitting criterion | Gini / Entropy (classification), Variance (regression) |
| Interpretability | Very high (can visualize rules) |
| Weakness | Prone to overfitting, high variance |

**Key Takeaways:**
- Decision trees are **highly interpretable** — you can trace every prediction
- Without pruning/depth limit, trees **overfit** perfectly to training data
- Trees are the building blocks of powerful ensemble methods (Random Forest, Gradient Boosting)
- Feature scaling is **not required** for tree-based methods